In [3]:
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
np.random.seed(0); plt.rcParams["figure.figsize"] = (8, 3.6)
df = pd.read_parquet("../inputs/cartera.parquet")

ArrowKeyError: A type extension with name pandas.period already defined

# A. Exploración de las variables de interes

Describan su ramo y exploren **solo** las variables que importan: la distribución de número de sinsitrsos, exposición y monto de siniestro. Reporten los descriptivos básicos (media, mediana, forma de la cola). No hagan un tour por todo el dataset: enfóquense.

In [2]:
print("Pólizas:", len(df), "| Columnas:", len(df.columns))
print("  frecuencia : num_siniestros, exposicion")
print("  severidad  : monto_promedio_siniestro (ground-up)")
print("  póliza     : deducible, suma_asegurada, coaseguro (para transformar)")

NameError: name 'df' is not defined

In [ ]:
# Exploración enfocada en componentes de la prima pura
 
# Filtrar pólizas con siniestros para estimar E[X] (severidad)
con = df[df.num_siniestros > 0]
 
# Tasa de frecuencia siniestral: proporción de pólizas con ≥1 siniestro
print(f"Tasa de frecuencia siniestral: {(df.num_siniestros>0).mean():.1%}")
 
# Exposición total en años-póliza: normaliza las métricas en el tiempo
print(f"Exposición total              : {df.exposicion.sum():,.0f} años-póliza")
 
# Severidad (E[X]): monto promedio por siniestro en pólizas siniestradas
print(f"Severidad (monto_promedio)    → media {con.monto_promedio_siniestro.mean():,.2f} "
      f"mediana {con.monto_promedio_siniestro.median():,.0f} ")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
 
# Frecuencia N: variable discreta
# Muestra la distribución de conteos de siniestros para revisar si hay Poisson o sobredispersión
ax[0].bar(*np.unique(df.num_siniestros, return_counts=True), color="#2C3E7F", edgecolor="#1A232A", alpha=0.85)
ax[0].set_title("Frecuencia Siniestral (N)"); ax[0].set_xlabel("Número de siniestros")
ax[0].grid(axis='y', alpha=0.3)
 
# Severidad X - Histograma: muestra densidad de frecuencia en rangos de montos
# Útil para ver dónde se concentran los valores y detectar multimodalidad
ax[1].hist(con.monto_promedio_siniestro, bins=int(np.sqrt(len(con))), color="#16A085", edgecolor="#0D3D2F", alpha=0.8)
ax[1].set_title("Severidad (X) - Histograma"); ax[1].set_xlabel("Monto de siniestro")
ax[1].grid(axis='y', alpha=0.3)
 
# Severidad X - Violin plot: muestra cuartiles, mediana y forma de la distribución
# Mejor para comparar asimetría y cola pesada; ve directamente la estructura del riesgo
parts = ax[2].violinplot([con.monto_promedio_siniestro], orientation='vertical', showmedians=True, showextrema=True)
for pc in parts['bodies']:
    pc.set_facecolor("#16A085")
    pc.set_alpha(0.7)
ax[2].set_title("Severidad (X) - Distribución"); ax[2].set_ylabel("Monto de siniestro")
ax[2].set_xticks([1]); ax[2].set_xticklabels(['Severidad'])
ax[2].grid(axis='y', alpha=0.3)
 
plt.tight_layout(); plt.show()

# B. Severidad

- Ajusten varias distribuciones (lognormal, gamma, Weibull, Pareto…) y **elijan una con criterio**: AIC **y** el comportamiento de la **cola** (no solo el número).
- Consideren las **transformaciones** de la póliza: distingan la severidad **ground-up** (el daño real) de la que **paga la aseguradora** tras aplicar deducible etc. Recuerden lo de la Sesión 3: ajustar sobre datos ya transformados sesga.


In [ ]:

#ANÁLISIS POR EL AIC

# Nombramos para poder llamar al monto promedio de los siniestros registrados
x = con.monto_promedio_siniestro.values

# Definimos el Akaike Information Criterior
#Intente equilibrar que tan bien se ajustan los datos al modelo
def aic(dist, pr, d): return 2*len(pr) - 2*np.sum(dist.logpdf(d, *pr))

#Distribuciones a considerar para el ajuste de los datos
candidatas = {"Lognormal": stats.lognorm, "Gamma": stats.gamma,
        "Weibull": stats.weibull_min, "Pareto": stats.pareto}
ajustes, filas = {}, []
for nom, dist in candidatas.items():
    #.fit nos funciona para estimar los parámetros e ls distr. de acuerdo nuetsros datos
    # floc=0 porque no existen montos negativos
    pr = dist.fit(x, floc=0); ajustes[nom] = (dist, pr)
    filas.append({"Distribución": nom, "AIC": round(aic(dist, pr, x), 1)})
tabla = pd.DataFrame(filas).sort_values("AIC").reset_index(drop=True)
#Se finaliza con la distribució con menor AIC, pues a menor valor, mejor es el modelo
mejor = tabla.iloc[0]["Distribución"]
print(tabla.to_string(index=False)); print(f"\n→ La mejor opción por AIC: {mejor}")

In [ ]:
# ANÁLISIS POR LA COLA

#Para poder visualizar la concordancia de las gráficas con nuestrso datos, asignamos colores para diferenciar
col = {"Lognormal":"#17A623","Gamma":"#E4953A","Weibull": "#4C7FD1" ,"Pareto":"#CF312B"}

#Muestra el percentil en el que se encuentra el monto de siniestro que deja aproximadamente al 99% de los datos por debajo de él
p99 = np.percentile(x, 99); grid = np.linspace(x.min(), p99, 400)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(x, bins=70, density=True, range=(x.min(), p99), color="#d9d9d9", edgecolor="white")

for nom,(d,pr) in ajustes.items(): ax[0].plot(grid, d.pdf(grid,*pr), color=col[nom], lw=2, label=nom)
ax[0].set_title("Densidad ajustada"); ax[0].legend(fontsize=8)

xs = np.sort(x); S_emp = 1 - np.arange(1,len(xs)+1)/len(xs)
ax[1].plot(xs, S_emp, ".", ms=2, color="black", label="empírica")

for nom,(d,pr) in ajustes.items(): ax[1].plot(xs, d.sf(xs,*pr), color=col[nom], lw=2, label=nom)
ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].set_ylim(1e-4,1)

#Título de gráfica de la cola de supervivencia
ax[1].set_title("Cola (supervivencia log-log)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
dist_sev, pr_sev = ajustes[mejor]

print(f"E[X] ground-up (modelo {mejor}) = {dist_sev.mean(*pr_sev):,.0f}")
print("Donde el comportamiento de la empirica tiene un comportamiento similar a la Log-normal, es decir, con una cola pesada")

In [ ]:
# Severidad transformada que paga la aseguradora

def transformacion(x, deducible, coaseguro, SA):
    return np.minimum(np.maximum(x - deducible, 0) * (1 - coaseguro), SA)

pagado = transformacion(con.monto_promedio_siniestro.values,
                     con.deducible.values, con.coaseguro.values, con.suma_asegurada.values)
print(f"E[X] ground-up   : {con.monto_promedio_siniestro.mean():,.0f}")
print(f"E[X] transformada: {pagado.mean():,.0f}")
print(f"La póliza absorbe ~{1 - pagado.mean()/con.monto_promedio_siniestro.mean():.1%} de la severidad "
      f"(sobre todo por el deducible).")

# C. Frecuencia

- Ajusten **todas** las opciones que vimos —Poisson y binomial negativa— y **diagnostiquen**: índice de dispersión, sobredispersión, exceso de ceros (quasi-Poisson/ZIP a nivel diagnóstico).
- **Ustedes deciden** cuál queda mejor con base en el diagnóstico; no asuman Poisson de entrada.
- Usen correctamente la **exposición**: la tasa es total de siniestros entre total de exposición, no un promedio simple.

In [ ]:
# ── Poisson, quasi-Poisson y binomial negativa (con offset) ──────────
y = df.num_siniestros.values
off = np.log(df.exposicion.values)
Xc = np.ones((len(df), 1))

pois  = sm.GLM(y, Xc, family=sm.families.Poisson(), offset=off).fit()
quasi = sm.GLM(y, Xc, family=sm.families.Poisson(), offset=off).fit(scale="X2")
nb    = sm.NegativeBinomial(y, Xc, offset=off).fit(disp=0)

lam_hat = np.exp(pois.params[0])          # tasa por año-póliza
phi = quasi.scale
print(f"Tasa λ̂ (siniestros por año-póliza) = {lam_hat:.4f}")
print(f"Índice de dispersión (Var/media)   = {y.var()/y.mean():.3f}")
print(f"Factor de dispersión φ (quasi)     = {phi:.3f}   (>1 → sobredispersión)")
print(f"\nAIC Poisson           = {pois.aic:,.0f}")
print(f"AIC Binomial negativa = {nb.aic:,.0f}")
print(f"→ Gana: {'Binomial negativa' if nb.aic < pois.aic else 'Poisson'}")

In [ ]:
# ── Tasa de frecuencia: total siniestros / total exposición (NO promedio simple) ──
tasa_global = y.sum() / df.exposicion.sum()
print(f"Tasa de frecuencia global = Σsiniestros / Σexposición = {tasa_global:.4f}")
print(f"(vs. promedio simple de tasas individuales, que estaría sesgado por pólizas con poca exposición)")
print(f"Coincide con λ̂ del GLM Poisson: {lam_hat:.4f}  ✓" if abs(tasa_global - lam_hat) < 1e-6 else "⚠ revisar")

In [ ]:
# ── Diagnóstico de exceso de ceros + observado vs. esperado ──────────
mu_i = lam_hat * df.exposicion.values
ceros_obs = (y == 0).mean(); ceros_esp = np.exp(-mu_i).mean()
print(f"P(N=0) observada = {ceros_obs:.3f}  |  esperada Poisson = {ceros_esp:.3f}")

kmax = y.max(); ks = np.arange(0, kmax+1)
obs = np.bincount(y, minlength=kmax+1) / len(y)
esp_pois = np.array([stats.poisson(mu_i).pmf(k) for k in ks]).mean(axis=1)
plt.bar(ks-0.2, obs, width=0.4, color="#4C7FD1", label="observado")
plt.bar(ks+0.2, esp_pois, width=0.4, color="#E4953A", label="esperado Poisson")
plt.xlabel("siniestros k"); plt.ylabel("proporción"); plt.legend(); plt.show()
print("Decisión: hay sobredispersión (φ>1) y la NB baja el AIC → elijo BINOMIAL NEGATIVA.")

In [ ]:
# ── ZIP a nivel diagnóstico ───────────────────────────────────────────
from statsmodels.discrete.count_model import ZeroInflatedPoisson

zip_mod = ZeroInflatedPoisson(y, Xc, exog_infl=Xc, offset=off).fit(disp=0)

print(zip_mod.summary())
print(f"\nAIC Poisson           = {pois.aic:,.0f}")
print(f"AIC Binomial negativa = {nb.aic:,.0f}")
print(f"AIC ZIP               = {zip_mod.aic:,.0f}")

# Probabilidad de inflación estimada (proporción de "ceros estructurales")
pi_infl = 1 / (1 + np.exp(-zip_mod.params[-1]))  # logit-inverso del parámetro de inflación
print(f"\nProporción estimada de ceros por inflación (π): {pi_infl:.3f}")

In [ ]:
# ── Decisión final integrando los 3 modelos ──────────────────────────
print("Resumen diagnóstico de frecuencia:")
print(f"  Índice de dispersión Var(N)/E(N) = {y.var()/y.mean():.3f}")
print(f"  Factor de dispersión φ (quasi)    = {phi:.3f}")
print(f"  P(N=0) obs = {ceros_obs:.3f} | esperada Poisson = {ceros_esp:.3f} "
      f"(diferencia: {ceros_obs-ceros_esp:+.3f})")
print(f"  π estimada de inflación en ZIP    = {pi_infl:.3f}")
print(f"\n  AIC → Poisson: {pois.aic:,.0f} | NB: {nb.aic:,.0f} | ZIP: {zip_mod.aic:,.0f}")

modelos = {"Poisson": pois.aic, "Binomial Negativa": nb.aic, "ZIP": zip_mod.aic}
mejor_modelo = min(modelos, key=modelos.get)
print(f"\n  → Modelo con menor AIC: {mejor_modelo} (diferencia de solo "
      f"{abs(nb.aic - zip_mod.aic):.0f} puntos vs. BN, no es concluyente)")
print("  Se elige BINOMIAL NEGATIVA como modelo final: la sobredispersión es leve")
print("  (φ=1.04) y el exceso de ceros observado vs. esperado es prácticamente nulo,")
print("  por lo que no hay evidencia sustantiva de una población de ceros estructurales")
print("  pese a que el AIC del ZIP sea marginalmente menor.")

# D. Simulación de la pérdida agregada S Version A

Con la severidad y la frecuencia que eligieron, simulen S (el algoritmo de la Sesión 7), en **dos versiones**:

- **S ground-up:** con la severidad del daño real.
- **S transformada:** aplicando deducible y suma asegurada (lo que realmente paga la aseguradora).

Para cada versión reporten la **distribución de S**, la **prima pura** E[S] y los cuantiles **VaR y TVaR** al 99%. Comparen las dos: **¿cuánto de la pérdida absorbe la póliza?** ¿Cuál es la que sirve para la prima del producto y por qué?

In [ ]:
import matplotlib.pyplot as plt

# Función matemática de la póliza (aplica deducible, coaseguro y límite)
def transformar(x, deducible, coaseguro, SA):
    return np.minimum(np.maximum(x - deducible, 0) * (1 - coaseguro), SA)

# ——— Simulación del modelo colectivo (ground-up y transformada) ———
Lambda = lam_hat * df.exposicion.sum()  # siniestros esperados en la cartera
r_agg = Lambda / (phi - 1)              # dispersión agregada (Var = phi * Lambda)
p_agg = r_agg / (r_agg + Lambda)
frecuencia_agg = stats.nbinom(r_agg, p_agg)

ded = df.deducible.values; sa = df.suma_asegurada.values; coas = df.coaseguro.values
M = 2000 # Simulamos 2000 escenarios 
Sg = np.empty(M); Sp = np.empty(M)

for m in range(M):
    Ntot = frecuencia_agg.rvs() # Cuántos choques hubo en el año
    xg = dist_sev.rvs(*pr_sev, size=Ntot) # Cuánto costó cada choque
    pol = np.random.randint(0, len(df), Ntot) # A qué póliza le tocó el choque
    xp = transformar(xg, ded[pol], coas[pol], sa[pol]) # Lo que paga la aseguradora
    
    Sg[m] = xg.sum() # Suma total Ground-up
    Sp[m] = xp.sum() # Suma total Transformada

# ——— Resultados y Gráficas ———
def resumen(S):
    v = np.percentile(S, 99) # VaR al 99%
    return S.mean(), v, S[S > v].mean() # TVaR al 99%

print(f"Siniestros esperados en la cartera (Λ) = {Lambda:,.0f}\n")
print(f"{'Versión':14}{'E[S] (prima pura)':>20}{'VaR99':>16}{'TVaR99':>16}")
for nom, S in [("Ground-up", Sg), ("Transformada", Sp)]:
    e, v, t = resumen(S)
    print(f"{nom:14}{e:>20,.0f}{v:>16,.0f}{t:>16,.0f}")

print(f"\nLa póliza (deducible+límite) reduce la prima pura de la cartera un "
      f"{1 - Sp.mean()/Sg.mean():.1%}: esa es la S que sirve para tarificar el producto.")

# Gráfica
plt.figure(figsize=(10, 5))
plt.hist(Sg, bins=40, alpha=0.6, color="#8FA0C8", label="S ground-up")
plt.hist(Sp, bins=40, alpha=0.6, color="#17A69B", label="S transformada (paga la aseguradora)")
plt.xlabel("Pérdida agregada de la cartera S")
plt.legend()
plt.show()